# Resumen ejecutivo

Este proyecto desarrolla una red de nodos sismográficos IoT con adquisición triaxial y backend MQTT+MySQL, orientada al monitoreo de vibraciones estructurales en la banda 0.5–30 Hz. Durante este corte se priorizó la verificación funcional de hardware (v0.0.1 THT y v0.0.2 SMD), la puesta en marcha de la base de datos y la consolidación del firmware para adquisición cruda y publicación por bloques. Se identificaron restricciones energéticas y de reconexión asociadas al uso de Li-ion y deep-sleep en ESP8266, que guían las decisiones de regulación y de diseño de ciclos de trabajo. El objetivo inmediato es cerrar mediciones en banco (consumo, autonomía y latencia E2E) y validar cobertura Wi-Fi para el piloto.

## Avances clave

- Diseño de placa de desarrollo v0.0.1 (THT) y v0.0.2 (SMD).
- Prototipo físico para verificación dimensional y ensamble.
- Arquitectura de la base de datos MySQL.
- Análisis de energía y batería, con dificultades detectadas.
- Prototipo de firmware con timers/ISR para muestreo periódico (FreeRTOS).

## Riesgos activos

- Autonomía _real vs objetivo_
- *Deep sleep* con reset 
- Tensión de la pila de Li-ion difiere con $V_{cc}$ de alimentación del microcontrolador
- Reconexión robusta tras sleep.

## Próximos hitos

- Validación energética en banco
- Prototipo físico de la placa de desarrollo v0.0.2 (SMD)
- Inserción bulk y particionado en DB.


# Contexto y objetivos  
Este piloto aborda el monitoreo de vibraciones estructurales en banda **0.5-30 Hz** mediante una red de nodos IoT que **adquieren y publican datos crudos** para su procesamiento en servidor. Objetivos medibles del corte: (i) adquisicion estable a **200 Hz** y publicacion por bloques; (ii) exactitud $\leq 10 \%$ tras calibracion estatica de la IMU; (iii) disponibilidad de ingesta continua (MQTT$\rightarrow$Node-RED$\rightarrow$MySQL) y consultas por ventana temporal en dashboard.


# Alcance y trazabilidad

Esta iteración se centra en **adquirir y publicar datos crudos** de aceleración triaxial desde nodos ESP8266 con IMU MPU6050, en la banda 0.5–30 Hz. El procesamiento (filtrado, espectros, KPIs) se realiza **en el servidor**, dejando al nodo la mínima responsabilidad: medir, empaquetar y publicar por bloques con marca de tiempo (`ts_ms`, `fs_hz`, `seq`). La base de datos MySQL debe aceptar ingesta idempotente y consultas por ventana temporal; el dashboard expone disponibilidad por nodo y series para diagnóstico.

Quedan **fuera de alcance** del presente corte la instalación definitiva en obra, la optimización energética con conversión buck-boost y las alarmas avanzadas; se documentan como pendientes con riesgos asociados. La validación prioriza: estabilidad de muestreo, reconexión tras deep-sleep y consistencia de la ingesta (sin duplicados) end-to-end.

## Trazabilidad de requerimientos actuales

| Requerimiento (qué)                         | Artefacto (con qué)                         | Prueba/verificación (cómo)                         | Estado     |
|----------------------------------|---------------------------------------------|---------------------------------------------------|------------|
| Muestreo estable a 200 Hz        | Firmware con timer+ISR y lectura I2C | Desvío < 1 % en periodo y conteo en 30 min continuos | En curso   |
| Publicación por bloques (QoS1)   | MQTT tópicos `sismo/<site>/<nodeId>/data`   | Recepción íntegra y ordenada de N bloques en broker | En curso   |
| Reconexión tras deep-sleep       | GPIO16->RST y arranque idempotente | `wake->publish` p95 ≤ 3 s; `status`/LWT correcto     | Pendiente  |
| Ingesta idempotente en DB        | Tabla `mediciones` con UNIQUE  | `INSERT ... ON DUPLICATE KEY` sin duplicados        | OK/Parcial |
| Consulta por ventana temporal    | Índice `(nodo_id, measured_at)`             | SELECT rango con latencia aceptable (p50/p95)       | OK/Parcial |
| Trazabilidad de pines/IMU        | Pinout efectivo y calibración  | Lectura estable + offset corregido en reposo        | En curso   |
| Cobertura Wi-Fi en sitio piloto  | v0.0.2 con keep-out RF y plan de medición  | RSSI medio y pérdida < 1 % en 30 min                 | Planificado |


**Criterio de aceptación del corte:** firmware capaz de medir y publicar bloques a 200 Hz con `ts_ms/fs_hz/seq`, ingesta idempotente operativa en MySQL y reconexión funcional tras deep-sleep (jitter de muestreo, `wake->publish`, pérdida de mensajes).


# Arquitectura general (E2E)

## Diagrama de bloques (visión actual)

![alt text](Pictures/Diagrama1.png)

La solución se organiza en una tubería simple y robusta: cada nodo (ESP8266 + MPU6050) adquiere aceleración triaxial, agrega metadatos mínimos (ts_ms, fs_hz, seq, n, temp_c, vbat) y publica bloques crudos via MQTT hacia un broker Mosquitto. Un flujo en Node-RED valida el esquema, transforma y persiste en MySQL. Todo el posprocesamiento (filtrado, espectros, alarmas) se ejecuta en el servidor, manteniendo el firmware liviano.

El acoplamiento es débil y observable: los nodos usan QoS1 y LWT retenido para señalizar estado; al salir de deep-sleep el arranque es idempotente (re-sincroniza NTP, repone seq y reanuda publicación). El backend aplica inserción idempotente (UNIQUE por nodo_id, measured_at, seq) para tolerar reintentos, y el dashboard consulta por ventanas temporales indexadas. La comunicación va cifrada (TLS en MQTT y MySQL) y con usuarios de privilegios mínimos.

**Flujo lógico:** Nodos $\to$ MQTT (broker) $\to$ Node-RED (validación y SQL) $\to$ MySQL (persistencia) $\to$ Dashboard (consulta).

# Selección de plataforma MCU y módulos Wi-Fi  

## Fuentes (datasheets)

* ESP8266EX (Espressif): [https://www.alldatasheet.com/datasheet-pdf/view/1148030/ESPRESSIF/ESP8266EX.html](https://www.alldatasheet.com/datasheet-pdf/view/1148030/ESPRESSIF/ESP8266EX.html)
* MPU-6050 (InvenSense): [https://www.alldatasheet.es/datasheet-pdf/view/517744/ETC1/MPU-6050.html](https://www.alldatasheet.es/datasheet-pdf/view/517744/ETC1/MPU-6050.html)

## ESP8266 (ESP-07/ESP-12)

El ESP8266 integra CPU Tensilica LX106 @80–160 MHz, radio 802.11b/g/n 2.4 GHz, TCP/IP embebido, SPI/I2C/UART, timers y GPIO suficientes para un nodo de adquisición sísmica ligera. Para nuestra necesidad (muestreo 100–200 Hz y publicación MQTT), aporta capacidad de cómputo y conectividad de sobra, con un ecosistema maduro (Arduino core/RTOS), bajo costo y buena disponibilidad.

**Funciones principales:** Wi-Fi STA/AP, pila TCP/IP, TLS por software, interrupciones GPIO, timers HW, deep-sleep con wake en GPIO/RTC.

**Consumos típicos (orden de magnitud, dependientes de firmware y AP):**

* TX/RX Wi-Fi con ráfagas: picos ~200 mA; promedio en ráfagas 70–120 mA.
* Modem-sleep (CPU activa, radio duty-cycled): ~15–20 mA.
* Light-sleep (CPU en bajo consumo): ~0.8–2 mA.
* Deep-sleep: ~10–25 $\mu$A (despierta por GPIO/RTC externo).
  Estas cifras guían el dimensionamiento de batería y duty-cycle; se recomiendan mediciones reales en banco para validar.

**Módulo elegido (ESP-07 frente a ESP-01):** migramos de ESP-01 a ESP-07/ESP-12 por: (i) más GPIO útiles (I2C + INT de la IMU sin conflictos), (ii) mejor desempeño RF y keep-out claro para antena, y (iii) opción de antena externa (u.FL en algunas variantes) para ensayos de cobertura en edificio. Esto reduce riesgos de saturación de pines y mejora la robustez del enlace.

**Por qué no ESP32 (por ahora):** ESP32 agrega BT/BLE, más RAM/CPU y periféricos, pero incrementa costo y consumo en reposo. Dado que el ancho de banda de señal sísmica de interés es 0.5–30 Hz y la tasa de publicación es modesta, el ESP8266 cubre holgadamente requisitos con menor consumo en deep-sleep y menor BOM. Dejamos ESP32 como alternativa si más adelante requerimos: doble núcleo para procesamiento local, ADCs internos de mayor calidad, o cifrado TLS con mayores márgenes.

## MPU-6050 (acelerómetro+giroscopio 6-ejes)

El MPU-6050 integra acelerómetro y giróscopo triaxiales con **ADCs de 16 bits**, DMP, FIFO de 1024 B e interfaz I2C (400 kHz). Rango programable $\pm$ **2/4/8/16 g** y ±250/500/1000/2000 °/s. La interrupción `INT` permite muestreo cronometrado y lectura por ráfagas.

**Consumos (datasheet):**

* Acel+gyro+DMP activos: ~3.9 mA.
* Solo acelerómetro: ~500 $\mu$A.
* Acelerómetro en low-power: ~10 $\mu$A @1.25 Hz a ~140 $\mu$A @40 Hz.
* FIFO e INT reducen actividad del MCU al permitir lecturas en ráfagas.

**Justificación de elección:** amplia disponibilidad, costo bajo, documentación y ejemplos abundantes, y rendimiento suficiente para 100–200 Hz con ruido aceptable tras calibración de offsets/escala. El pin `INT` facilita sincronía de muestreo y el FIFO evita jitter por latencias de I2C/MQTT.

## Síntesis comparativa 

**ESP8266 (ESP-07/12) + MPU-6050** ofrece la mejor relación costo/consumo/capacidad para un nodo IoT sísmico: Wi-Fi estable, consumo en deep-sleep de orden decenas de $\mu$A, y sensor 6-ejes ~4 mA activo. ESP32 queda como “plan B” si más adelante necesitamos procesamiento embarcado o enlaces más exigentes; el ESP-07 se prefiere al ESP-01 por GPIO y performance RF (opción de antena externa) que simplifican el cableado de I2C+INT y los ensayos de propagación.

# Micro investigación de mercado

El monitoreo de vibraciones en la banda 0.5–30 Hz tiene dos ecosistemas cercanos pero distintos: **SHM civil/sísmico** y **PdM industrial**. El primero prioriza baja frecuencia real (hasta DC o 0.5 Hz), autonomía y robustez ambiental; el segundo privilegia diagnósticos de maquinaria y suele cortar por debajo de 1–3 Hz. En este contexto, los acelerómetros MEMS modernos de bajo ruido habilitan nodos IoT triaxiales de costo contenido y consumo moderado, mientras que las soluciones piezoeléctricas IEPE siguen dominando cuando se requiere precisión metrológica de alto nivel.

## Hallazgos clave
- **Compatibilidad de banda**: muchos transmisores 4–20 mA industriales fallan por debajo de 3–10 Hz; no sirven para 0.5–30 Hz. En cambio, MEMS triaxiales específicos de baja frecuencia (p. ej., líneas tipo ADXL35x/derivados) sí cubren la banda objetivo. 
- **Nodos COTS**: existen equipos SHM inalámbricos con rango útil desde 0.5 Hz y pisos de ruido <100 μg/√Hz; son robustos pero **caros** y, a menudo, con radio/protocolo propietarios (LoRaWAN/protocolos 2.4 GHz). 
- **Argentina (ecosistema)**: hay integradores locales para soluciones “llave en mano” y distribuidores de sensores sísmicos de baja frecuencia (v.g., VibraSens vía representante local), además de servicio/calibración con trazabilidad internacional. 
- **Backend**: MQTT es estándar para telemetría; **MySQL** funciona para prototipo.
- **Estrategia**: para este proyecto académico con control de pila y costo, conviene **Integrar** (sensores adecuados + nodos propios + MQTT + DB) en lugar de Comprar COTS o Desarrollar desde cero ASIC/PCB de alto NRE. 

## Implicancias para nuestro diseño
Para capturar 0.5–30 Hz con SNR adecuado y mantener autonomía, se justifica continuar con **nodos IoT MEMS triaxiales** y publicación vía **MQTT**; mantener **MySQL**. En Argentina existe oferta para: (a) sensores de baja frecuencia, (b) integración LoRaWAN/MQTT, y (c) **calibración** con trazabilidad, lo que viabiliza escalamiento y validación metrológica local en siguientes cortes. 


# Diseño HW — Placa v0.0.1 (THT)

## Descripción General

Se diseno una primera revision orientada a verificacion funcional. La alimentacion se resolvio con batería Li-ion, diodo serie y LDO AMS1117-3.3, suficiente para banco pero limitada en eficiencia y margen de caida. La logica de arranque incluye pulsadores de BOOT y RESET con EN por pull-up. La IMU MPU6050 se conecta por I2C y expone INT al MCU. Se dispone de senalizacion mediante cuatro LEDs y de headers para UART y MPU6050.

* **Formato:** 61.98 $\times$ 33.02 mm, 4 orificios de montaje.
* **Topologia:** batería Li-ion $\to$ diodo serie $\to$ **LDO AMS1117-3.3** $\to$ rail 3V3 para **ESP-07** y **GY-521**.
* **Senalizacion:** cuatro LEDs (Wi-Fi, datos, aux1, aux2) con resistencias limitadoras.
* **Control:** pulsadores de **BOOT** y **RESET**; **EN** con pull-up (R3).
* **Interfaz:** header **UART** para programacion y header **I2C**/INT para IMU.
* **Conectividad IMU:** `SCL/SDA`, `INT` y `AD0` cableados al conector J4 (modulo MPU6050).

## Bloques y esquemas 

![Figura 1. Esquematico eléctrico (v0.0.1).](Pictures/Vibranet%20v0.0.1%20-%20Esquematico%20electrico.jpeg){#fig:1v0.0.1}

1. **Alimentacion**: AMS1117-3.3 con **C1=47 $\mu$F** de bulk y ceramicos de desacople ($C_2$, $C_3$). Diodo **1N4007** en serie desde batería como solucion transitoria para reducir Vbat.
2. **Control de arranque**:
   * `BOOT` con $R_1=10 k\Omega$ a GND y pulsador a VCC para forzar modo flash cuando se requiera.
   * `RESET` con $R_2=10 k\Omega$, pulsador a GND y **EN** con $R_3=10 k\Omega$ a VCC.
3. **MCU + IO**: **ESP-07** con GPIOs mapeados a UART, I2C y LEDs ($R_4-R_7$).

![Figura 2. Plano de conexión y dimensiones.](Pictures/Vibranet%20v0.0.1%20-%20Plano%20de%20conexion.jpeg){#fig:2v0.0.1}

Ruteo de una cara con puentes; headers dedicados para **UART** y **MPU6050**. Se senalan keep-out y leyendas (autor, version).

## Pinout efectivo (para trazabilidad de firmware)

| Funcion          | Pin ESP8266   | Conector | Nota                         |
| ---------------- | ------------- | -------- | ---------------------------- |
| UART TX          | GPIO1/TXD0    | J2-TX    | Programacion/registro        |
| UART RX          | GPIO3/RXD0    | J2-RX    | Programacion/registro        |
| I2C SCL          | GPIO5         | J4-SCL   | Pull-ups en modulo GY-521    |
| I2C SDA          | GPIO4         | J4-SDA   | —                            |
| IMU INT          | GPIO14 (int1) | J4-INT   | Interrupciones de movimiento |
| LED Wi-Fi        | GPIO12        | D1 + R4  | Estado radio                 |
| LED Datos        | GPIO13        | D2 + R5  | Publicacion/actividad        |
| LED Aux 1        | GPIO15        | D3 + R6  | Uso general                  |
| LED Aux 2        | GPIO16        | D4 + R7  | Uso general                  |
| EN (chip-enable) | EN            | —        | Pull-up $R_3=10 k\Omega$     |
| RESET            | RST           | SW2      | Pulsador a GND               |
| BOOT             | GPIO0         | SW1/JP1  | Forzar modo programacion     |

## Riesgos e impactos

| Riesgo                      | Impacto                   |
| --------------------------- | ------------------------- |
| Dropout con LDO + diodo     | Resets en TX              |
| Falta de desacople correcto | Ruido en IMU/MCU          |                             
| Sin medicion de corriente   | Autonomia incierta        |
| Proteccion insuficiente     | Sensibilidad a ESD/fallos |

## Registro fotográfico (referencias)

* **Vista superior**: ubicacion de ESP-01, GY-521, LED y controles.
* **Vista inferior**: batería y cargador/proteccion con cableado.
* **Vista lateral**: apilado de modulos, disipador en ESP-01, accesos a headers.
* **Vista isometrica**: vista general de la disposicion del prototipo.

\begin{figure}[ht]
  \centering

  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Vibranet v0.0.1 - Vista Inferior Angular.jpeg}}
    \caption{Vista inferior angular}
  \end{subfigure}
  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Vibranet v0.0.1 - Vista Inferior Angular.jpeg}}
    \caption{Vista inferior angular}
  \end{subfigure}

  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Vibranet v0.0.1 - Vista Superior.jpeg}}
    \caption{Vista superior}
  \end{subfigure}
  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Vibranet v0.0.1 - Vista Isometrica Superior.jpeg}}
    \caption{Vista isométrica superior}
  \end{subfigure}

  \caption{Vibranet v0.0.1: bloque 2×2 de vistas.}
  \label{fig:vib001}
\end{figure}

_Figura: Vibranet v0.0.1 (vistas del prototipo THT)._


# Problemática de energia — Batería Li-ion

La red de nodos requiere **autonomia electrica** para operar sin red fija. La fuente elegida es una celda Li-ion de factor de forma compacto ($\leq 50\times 50 \ \mathrm{mm}$, espesor $< 7.5 \ \mathrm{mm}$). El ESP8266 opera entre **2.5-3.6 V** y presenta cuatro modos de potencia (activo, modem-sleep, light-sleep y deep-sleep). Segun el [datasheet del ESP8266EX](https://documentation.espressif.com/0a-esp8266ex_datasheet_en.pdf) , el consumo tipico es: $\mathrm{deep\text{-}sleep} \approx 20 \mu \mathrm{A}$, $\mathrm{light\text{-}sleep} \approx 0.9  \mathrm{mA}$, $\mathrm{modem\text{-}sleep} \approx 15 \mathrm{mA}$, y en activo con radio el promedio ronda **decenas de mA** con picos de **>100 mA** en TX/RX (ver Tabla 3-4 "Power Consumption by Power Modes" y Tabla 1-1 "Specifications").

El consumo del modulo MPU6050 se debe suponer hasta realizarse mediciones. **Supuesto**: la IMU **MPU6050** anade **$\approx$ 3-4 mA** en adquisicion continua y **< 10 $\mu$A** en modo low-power accel. Estos valores se validaran en banco.

**Implicancia:** la autonomia no la determina el consumo de $20 \mu \mathrm{A}$ en deep-sleep, sino el **duty cycle real** entre medir, transmitir y dormir. Ademas, el **rango de tension de la celda Li-ion ($\approx 4.2 \to 3.0 V$)** no es directamente compatible con la ventana $2.5-3.6 V$ del ESP8266 si no se condiciona la tension.


## Restricciones de diseno

1. **Tensión**

   * Batería Li-ion: 4.2 V (cargada) a ~3.0 V (descargada).
   * ESP8266: 2.5-3.6 V.
   * **Conclusión:** se requiere regulacion.

     * Un **diodo en serie** ($\approx 0.7$ V) reduce 4.2 -> 3.5 V, pero **colapsa** a ~2.3 V cuando la celda cae a 3.0 V, **bajo el minimo** del ESP8266. No es solucion de produccion.
     * Un **LDO** a 3.3 V desaprovecha la cola de la batería (cuando Vin < 3.3 V entra en dropout).
     * Un **buck** a 3.3 V es eficiente por arriba de 3.3 V, pero **no regula** cuando Vin < 3.3 V.

2. **Ciclo de operación**
   Definimos un ciclo con tres estados:
   * **Activo (adquisicion + TX)**: ESP8266 activo con radio, IMU activa.
   * **Idle/light-sleep**: CPU o perifericos pausados, radio apagado.
   * **Deep-sleep**: solo RTC operativo; en ESP8266 el **wake implica reset** del MCU.

3. **Recuperación post-sleep**
   El **deep-sleep reinicia** el ESP8266: se debe **reconstruir estado** (NTP, MQTT, `seq`, reconfiguracion IMU) de forma **idempotente**.

## Marco de calculo de autonomia

Para un ciclo de duracion  $T=t_a+t_l+t_d$ con corrientes $I_a, I_l, I_d$:

Considerando:

* $T$: duracion total de un ciclo.
* $t_a$: tiempo en **estado activo** (adquisicion + TX/RX).
* $t_l$: tiempo en **light-sleep** (CPU/perifericos en pausa, radio apagada).
* $t_d$ : tiempo en **deep-sleep** (solo RTC activo).
* $I_a$: **corriente media** en estado activo.
* $I_l$: **corriente media** en light-sleep.
* $I_d$: **corriente media** en deep-sleep.

$$
I_{\text{prom}}=\frac{I_a t_a + I_l t_l + I_d t_d}{T}
\quad\Rightarrow\quad
\text{Autonomia (h)}\approx \frac{C_{\text{mAh}}\cdot \eta}{I_{\text{prom}}}
$$

* **Capacidad efectiva** $C_{\text{mAh}}$: nominal $\times$ **factor de eficiencia** $\eta\in[0.8,0.9]$ para perdidas de regulador y temperatura.
* **Supuestos iniciales** (a validar):
  $I_a=120\text{ mA}$ promedio durante TX y lectura, $I_l=0.9\text{ mA}$, $I_d=20\ \mu\text{A}$.
  IMU activa suma ~4 mA en $I_a$ y ~10 $\mu$A en bajo consumo.

**Ejemplo de orden de magnitud (no compromiso de diseno):**
Si $t_a=2\text{ s}$, $t_l=8\text{ s}$, $t_d=50\text{ s}$ por ciclo de 60 s, entonces
$I_{\text{prom}}\approx \frac{(124)(2)+(0.9)(8)+(0.02)(50)}{60}\approx 4.5\text{ mA}$.
Con una celda **2000 mAh** y $\eta = 0.85  \to \approx 377 h$ teoricas. Pequenos cambios en $t_a$ o $I_a$ derrumban la cifra, de ahi la necesidad de medir.


# Diseño HW — Placa v0.0.2 (SMD)


## Descripción general

Revision orientada a **compactar** el diseno y mejorar **DFM/EMI** manteniendo la funcionalidad de la v0.0.1. Se migra a **SMD** con plano de masa continuo, keep-out para antena y headers accesibles para IMU y UART. La alimentacion continua con **LDO AMS1117-3.3 + diodo serie** como solucion de banco, en espera de migrar a **buck-boost 3.3 V** en la siguiente iteracion.

* **Formato:** **51.56 $\times$ 38.61 mm**, 4 orificios de montaje.
* **Topologia:** batería Li-ion $\to$ diodo serie $\to$ **LDO AMS1117-3.3** $\to$ rail 3V3 para **ESP-12** y conector **MPU6050**.
* **Keep-out RF:** zona superior libre de cobre y componentes sobre la antena integrada (serigrafia "KEEP-OUT ZONE").
* **Control:** pulsador **RESET**, pad para **BOOT/FLASH** y `EN` con pull-up.
* **Interfaz:** header **UART** de programacion y header **I2C/INT** para la IMU.
* **Senalizacion:** tres LEDs SMD para estado (Wi-Fi/datos/aux).

Vease **Cobertura y propagacion (plan de medicion)** para los ensayos de antena y cobertura asociados a esta revision.

## Bloques y esquemas

![Figura 3. Esquemático eléctrico (v0.0.2 SMD).](Pictures/Vibranet%20v0.0.2%20\(SMD\)%20-%20Esquematico%20electrico.jpeg){#fig:3v0.0.2}


1. **Alimentación**: **AMS1117-3.3** con **C3 bulk** y ceramicos de desacople cercanos; **diodo 1N4007** serie desde batería como atajo temporal para ajustar Vbat.
2. **RF/MCU**: **ESP-12 (ESP8266MOD)** con antena integrada Rainsun y keep-out superior; GPIO mapeados a UART, I2C y LEDs.
3. **Control de arranque**: `EN` con pull-up, `RESET` por pulsador; pads para `GPIO0` (modo flash).
4. **IMU**: conector para **MPU6050** con `SCL/SDA` y `INT`.

![Figura 4. Plano de conexión y dimensiones (v0.0.2 SMD).](Pictures/Vibranet%20v0.0.2%20\(SMD\)%20-%20Plano%20de%20conexion.jpeg){#fig:4v0.0.2}

Ruteo con **plano GND** en top y via-stitching perimetral; colocacion SMD en bancos compactos para LEDs y RC de control; headers verticales para IMU/UART; keep-out RF claramente marcado.

#### Pinout efectivo (para trazabilidad de firmware)

| Funcion          | Pin ESP8266 | Conector | Nota                         |
| ---------------- | ----------- | -------- | ---------------------------- |
| UART TX          | GPIO1/TXD0  | J3-TX    | Programacion/registro        |
| UART RX          | GPIO3/RXD0  | J3-RX    | Programacion/registro        |
| I2C SCL          | GPIO5       | J2-SCL   | Pull-ups en modulo IMU       |
| I2C SDA          | GPIO4       | J2-SDA   | —                            |
| IMU INT          | GPIO14      | J2-INT   | Interrupciones de movimiento |
| LED Wi-Fi        | GPIO12      | D? + R?  | Estado radio                 |
| LED Datos        | GPIO13      | D? + R?  | Publicacion/actividad        |
| LED Aux          | GPIO16      | D? + R?  | Uso general                  |
| EN (chip-enable) | EN          | —        | Pull-up                      |
| RESET            | RST         | SW1      | Pulsador a GND               |
| BOOT/FLASH       | GPIO0       | Pad/JP   | Forzar modo programacion     |



# Prototipo físico

## Descripción general

Modulo de adquisicion autonomo montado en placa perforada (protoboard FR-4) con **batería Li-ion plana** como sustrato mecanico. El conjunto integra **ESP8266 (ESP-01)**, **IMU MPU6050 (GY-521)**, **cargador/proteccion de batería Li-ion**, y **modulo DC-DC** para acondicionamiento de tension. Se incorporan **interruptor**, **pulsador de reset/usuario**, **LED de estado** y **test-pads** para depuracion.

## Componentes principales

| Conjunto                           | Funcion                         | Observaciones de montaje                                                  |
| ---------------------------------- | ------------------------------- | ------------------------------------------------------------------------- |
| ESP8266 ESP-01 con soporte impreso | MCU + Wi-Fi                     | Soporte 3D para alivio de tension del conector; disipador pasivo pequeno. |
| MPU6050 (GY-521)                   | Acelerometro/giroscopio         | Montado en el plano superior; acceso a SDA/SCL en header.                 |
| Cargador/protector Li-ion          | Carga 5 V y proteccion de celda | Entrada micro-USB; ubicado en cara inferior junto a la batería.           |
| DC-DC (ajustable)                  | Regulacion a 3.3 V              | Modulo comercial; ajuste en 3.30 +- 0.03 V.                               |
| Batería Li-ion tipo "flat"         | Fuente                          | Fijacion con cinta Kapton; cableado con alivio termico.                   |
| Diodo serie (temporal)             | Caida de ~0.7 V                 | Solucion transitoria para compatibilidad de tension.                      |
| Interfaz                           | Interruptor, pulsador, LED      | Acceso lateral; utiles para pruebas de banco.                             |

## Interconexión electrica (alto nivel)

* **Batería -> cargador/proteccion -> DC-DC 3.3 V -> rail logica (ESP8266 + IMU)**.
* **GPIO/UART**: SDA/SCL expuestos; UART accesible para programacion.

## Funcionalidad validada en banco

* Encendido estable a 3.3 V regulados.
* Lectura de IMU.
* Carga de batería por micro-USB y operacion en modo "cableado".

## Limitaciones actuales

* **Regulación**: uso de diodo de caida como solucion temporal; riesgo de subalimentacion cuando Vbat < 3.3 V + Vdrop.
* **Mecanica**: apilado con adhesivo caliente; se requiere placa SMD y separadores para rigidez y repetibilidad.
* **Medición de consumo**: sin shunt/INA dedicado; no hay perfil de corrientes por estado.
* **Protección**: sin TVS ni fusible rearmable en entrada; headers sin retencion.

## Proximos pasos de mejora

1. Migrar a **PCB v0.0.2 SMD** con plano GND continuo bajo IMU y "via-stitching".
2. Incorporar **montajes mecanicos**: tornilleria M2.5, separadores, y casquillos para desacoplar vibraciones.
3. Anadir **protecciones**: TVS en entrada USB, PTC 500 mA, y filtro LC en rail 3.3 V.

## Registro fotográfico (referencias)

* **Vista superior**: ubicacion de ESP-01, GY-521, LED y controles.
* **Vista inferior**: batería y cargador/proteccion con cableado.
* **Vista lateral**: apilado de modulos, disipador en ESP-01, accesos a headers.
* **Vista isometrica**: vista general de la disposicion del prototipo.

\begin{figure}[ht]
  \centering

  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Prototipo v1 - Vista Inferior.jpg}}
    \caption{Vista inferior}
  \end{subfigure}
  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Prototipo v1 - Vista Lateral 1.jpg}}
    \caption{Vista lateral 1}
  \end{subfigure}

  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Prototipo v1 - Vista Superior.jpg}}
    \caption{Vista superior}
  \end{subfigure}
  \begin{subfigure}{0.48\linewidth}
    \centering
    \includegraphics[width=\linewidth]{\detokenize{Pictures/Prototipo v1 - Vista Isometrica.jpg}}
    \caption{Vista isométrica}
  \end{subfigure}

  \caption{Prototipo v1: bloque 2×2 de vistas.}
  \label{fig:protov1}
\end{figure}

_Figura: Prototipo fisico v1 — vistas del ensamblado._


# Firmware (ESP8266/ESP32)  <!-- AJUSTADO -->

## Objetivo y alcance

El firmware temporiza la adquisicion con un **timer periodico** (200 Hz). La **ISR** solo encola un disparo; una tarea **lee la IMU (MPU6050) por I2C en *burst***, aplica **calibracion de offset** (guardada en NVS/EEPROM) y llena un **buffer de tamano fijo**. Al completar el bloque, otra tarea empaqueta y **publica por MQTT** con **QoS1**; si no hay red, **encola en SPIFFS** para reenvio. El payload incluye `ts_ms` (**marca de tiempo UTC en milisegundos del inicio del bloque**), `fs_hz`, `seq`, `n` y las series crudas por canal. En ESP8266, el *deep-sleep* despierta por **GPIO16$\rightarrow$RST**, por lo que el arranque rehidrata NTP/MQTT/IMU y `seq`. Adquirir datos crudos de la **MPU6050** y publicarlos sin procesamiento local, delegando el posprocesado en el backend. Veanse \S\S14 y 15 para la integracion con deep-sleep y la justificacion de `F_s`.

## Temporización y ejecución

* **Temporizador periodico**: genera una interrupcion cada $T=1/FS_{\text{HZ}}$ para disparar la toma de muestra.
* **ISR minima**: coloca un "trigger" en una cola.
* **Tarea de adquisición**: toma el "trigger", realiza **lectura burst I2C** del MPU6050, **aplica calibracion** y escribe la muestra cruda en un **ring buffer**.
* **Tarea de publicacion**: empaqueta **bloques de N muestras** y publica por MQTT (o los deja en cola local si no hay red).
* **Deep-sleep (ESP8266)**: `GPIO16 $\rightarrow$ RST` produce reset; en `setup()` se rehidrata NTP/MQTT/IMU y se continua la secuencia `seq`.

## Calibración del MPU6050 (compensación de offset)

**Intención:** remover sesgo de fabrica y de montaje.

1) Nodo inmovil 10-20 s. 

2) Capturar N muestras. 

3) Calcular medias por canal.
   Acel: `bias_a[i]=mean(acc[i])` ajustando Z a $\pm$ 1 g; Giro: `bias_g[i]=mean(gyro[i])`.
   Guardar en NVS/EEPROM junto con `temp_ref`. En operacion: `x_corr = x_raw − bias − k_temp*(temp − temp_ref)` si se calibra termicamente.
   **Criterio de aceptacion interno:** var(acc) < 0.01 g.

## MQTT: tópicos y payload 

* Tópicos:

  * `sismo/<site>/<nodeId>/data` (QoS1, no retenido)
  * `sismo/<site>/<nodeId>/status` (QoS1, retenido, LWT=offline)
  * `sismo/<site>/<nodeId>/cfg` (QoS1, retenido)

**`ts_ms` significa "timestamp en milisegundos desde epoca UNIX (UTC)"**. En modo por bloques, `ts_ms` es el **instante de la primera muestra** del bloque. La separacion temporal entre muestras del bloque se infiere con `fs_hz` o con `dt_us` opcional.

**Payload propuesto (bloques crudos, sin filtrado):**

```json
{
  "ts_ms": 1730158805123,
  "fs_hz": 200,
  "seq": 15231,
  "n": 100,
  "ax": [ ... 100 valores ... ],
  "ay": [ ... 100 valores ... ],
  "az": [ ... 100 valores ... ],
  "gx": [ ... 100 valores ... ],
  "gy": [ ... 100 valores ... ],
  "gz": [ ... 100 valores ... ],
  "temp_c": 27.5,
  "vbat": 3.92
}
```

Justificación de parametros:

- `ts_ms` y `fs_hz` permiten reconstruir el tiempo exacto de cada muestra en servidor.
 
- `seq` asegura idempotencia extremo a extremo junto al `UNIQUE(nodo_id, measured_at, seq)` en DB.

- `n` define tamano del bloque para decodificacion eficiente.

- `temp_c` habilita compensaciones termicas en analisis.

- `vbat` permite correlacionar eventos con caida de tension.

# Cobertura y propagación (plan de medicion)  <!-- NUEVO -->

La performance depende del entorno (hormigon, acero, geometria de planta). Con la v0.0.2 se respeta el **keep-out RF** de antena; aun asi, se realizara un **site-survey** en dos pasos: (1) recorrido estatico con un nodo portatil registrando **RSSI** y tasa efectiva de publicacion; (2) prueba dinamica en horarios pico midiendo **perdida** y **latencia "wake$\rightarrow$publish"** con *deep-sleep*. Criterios de accion: si **RSSI < -75 dBm** sostenido o **perdida > 1 %** en 30 min, se define **AP auxiliar** o **antena externa** (ESP-07) y se reubican nodos. El backend usara **status retenido + LWT** para mapear disponibilidad por ubicacion.


# Deep-sleep y "reset virtual" (ESP8266)

Es un **hecho tecnico** que en el microcontrolador ESP8266 el modo **deep-sleep** no "despierta" el programa donde quedo: **reinicia** el chip. El despertar se logra con la union **GPIO16 (D0) $\to$ RST**; al vencer el temporizador interno, **GPIO16 impulsa RST** y se produce un **reset por hardware** equivalente a un "software/virtual reset".

**Implicancias de diseno**

* **Re-hidratacion de estado:** al iniciar, el firmware debe reconstruir NTP, MQTT, configuracion de IMU y contadores (`seq`) de forma **idempotente**.
* **Persistencia minima:** guardar en RTC/flash ligera: `seq`, ultimo `ts`, flags de "shutdown limpio".
* **Tiempos de servicio:** medir y reportar `t_wake $\rightarrow$\rightarrow$\righta$\rightarrow$\rightarrow$row$ Wi-Fi $\rightarrow$\rightarrow$\rightarrow$\rightarrow$ publish (p95)`; los costos de asociacion Wi-Fi dominan el consumo si el ciclo de sueno es corto.
* **Senalizacion y LWT:** publicar `status=boot/deepsleep_wakeup` y configurar **LWT** para recuperacion en backend.

**Requisito eléctrico**

* Trazar **GPIO16 $\to$ RST** con retorno GND corto y mantener **pull-ups** de RST/EN segun datasheet. No conectar GPIO16 a otros perifericos.

**Prueba minima**

1. Programar `deepSleep(us)`, verificar pulso en RST desde GPIO16.
2. Comprobar re-inicio completo y publicacion de `status` en <= 3 s tras el reset.

En ESP8266, `deepSleep(us)` despierta por **pulso de GPIO16 a RST**, equivalendo a un **reset por hardware**. El arranque debe sincronizar NTP, reconectar MQTT publicando `status=deepsleep_wakeup`, restaurar `seq`, reconfigurar la IMU y enviar/barrer cualquier bloque encolado en SPIFFS.


# Backend y dashboard
Node-RED procesa la ingesta (validacion del esquema, construccion de SQL) y publica KPIs en un dashboard (disponibilidad por nodo, volumen por dia y graficos de aceleracion). La conexión a MySQL se realiza con **TLS** y usuarios de privilegios minimos.

#  Frecuencias de muestreo — justificación  <!-- NUEVO (consolidado) -->

Las vibraciones sismicas de interes estan en **0.5-30 Hz**. Se adopta **$F_s=200\,\text{Hz}$** por **sobre-muestreo >= 5x** respecto de $f_{\max}$ y margen para transitorios y jitter; Nyquist queda en 100 Hz. Publicar por **bloques** a 200 Hz mantiene resolucion y un volumen de datos razonable; cualquier filtrado/decimado se realiza en servidor.

## Justificación de frecuencias de muestreo

Las vibraciones sismicas estructurales de interes se concentran tipicamente entre **0.5 y 30 Hz**. Para capturarlas sin distorsion y con margen de analisis, se adopta un **factor de sobre-muestreo >= 5x** respecto de la frecuencia maxima de interes:

$$
F_s \ge k \cdot f_{\max}\quad \text{con}\quad k \in [5,10]
$$

Con $f_{\max}=30\,\text{Hz}$ y $k=5$ resulta $F_s \ge 150 \, \text{Hz}$. Se fija $F_s=200 \, \text{Hz}$ por las siguientes razones:

1. deja **banda util hasta 100 Hz** (Nyquist) para cubrir picos transitorios o impactos por encima de 30 Hz;
2. mejora la estimacion de espectros y tiempos caracteristicos al contar con mas muestras por ciclo;
3. aporta **robustez a jitter** del temporizador y a pequenas derivas del bus I2C sin perder cobertura;
4. mantiene un **coste de red razonable** cuando se publica en **bloques** (sin filtrado en el nodo), dejando todo el posprocesamiento al servidor.

Parametros operativos asociados:

* `FS_HZ = 200` define la frecuencia de adquisicion en el microcontrolador.
* `PUB_BLOCK_SAMPLES` controla el tamano del bloque publicado y, por ende, la **latencia** y el **ancho de banda** efectivo. Por ejemplo, `PUB_BLOCK_SAMPLES = 100` implica bloques de **0.5 s** a 200 Hz.

Este esquema garantiza resolucion suficiente para la banda 0.5-30 Hz con margen de analisis y sin carga de computo en el firmware, que solo **mide y publica** datos crudos.


# Riesgos y mitigaciones
Energia (dropout y autonomia), RF/cobertura (areas sombra), DB/retencion (tamano y tiempos de consulta), robustez mecanica. Mitigaciones: buck-boost 3.3 V, site-survey y/o AP auxiliar, particionado temporal, pads de test y protecciones ESD/PTC.

## Riesgos y mitigaciones

| Riesgo                       | Efecto            | Mitigacion                                                    |
| ---------------------------- | ----------------- | ------------------------------------------------------------- |
| Picos de TX > 200 mA         | Reset brown-out   | Capacitores cerca del SoC y del regulador, ruta de GND corta  |
| Dropout en LDO               | Apagado prematuro | Buck-boost o cutoff a Vbat que garantice 3.3 V estable        |
| Overhead de reconexion Wi-Fi | Aumenta $t_a$     | Persistir credenciales, aumentar DTIM, batch de publicaciones |
| Deriva de consumo IMU        | Baja autonomia    | Cambiar a modo low-power accel entre eventos                  |

# Plan de proxima iteracion
Energia (mediciones por estado y decision de regulador), cobertura (site-survey y acciones), firmware (cola local y secuencia persistente), DB (particiones y vistas), instalacion piloto acotada.

## Cronograma y plan de la siguiente iteracion

**Objetivos verificables**

* Crear un prototipo funcional de la placa de desarrollo v0.0.2 (SMD)
* Implementar cola local en firmware con reenvio y prueba de caida 10 min.
* Integrar TLS mutual en Mosquitto y validar ACLs por topico.
* Completar dashboard con espectro en ventana movil y alarmas.
* Ensayo de calibracion de 2 nodos y reporte de error con incertidumbre.



# Anexos

## Bibliografía y referencias

[1] Espressif Systems, "ESP8266EX Datasheet," 2019. [En linea]. Disponible: https://www.alldatasheet.com/datasheet-pdf/view/1148030/ESPRESSIF/ESP8266EX.html. 

[2] Espressif Systems, "ESP8266EX Technical Reference," v2.x. [En linea]. Disponible: https://docs.espressif.com/projects/esp8266-rtos-sdk/en/latest/. 

[3] Espressif Systems, "ESP8266 RTOS SDK Programming Guide." [En linea]. Disponible: https://documentation.espressif.com/en/home.

[4] TDK InvenSense, "MPU-6000 and MPU-6050 Product Specification," Rev. 3.4. [En linea]. Disponible: https://www.alldatasheet.es/datasheet-pdf/view/517744/ETC1/MPU-6050.html. 

[5] TDK InvenSense, "MPU-6050 Register Map and Descriptions," Rev. 4.2. [En linea]. Disponible: https://invensense.tdk.com/. 

[6] OASIS, "MQTT Version 3.1.1 Plus Errata 01," OASIS Standard, Dec. 2015. [En linea]. Disponible: https://docs.oasis-open.org/mqtt/mqtt/. 

[7] Eclipse Foundation, "Eclipse Mosquitto - Open Source MQTT Broker." [En linea]. Disponible: https://mosquitto.org/. 

[8] Node-RED Project, "Node-RED Documentation." [En linea]. Disponible: https://nodered.org/docs/. 

[9] Oracle, "MySQL 8.0 Reference Manual." [En linea]. Disponible: https://dev.mysql.com/doc/. 

[10] D. Mills, J. Martin, J. Burbank, y W. Kasch, "Network Time Protocol Version 4: Protocol and Algorithms Specification," RFC 5905, Jun. 2010. [En linea]. Disponible: https://www.rfc-editor.org/rfc/rfc5905. 

[11] Advanced Monolithic Systems, "AMS1117 Fixed 1A Low Dropout Linear Regulator," Datasheet. [En linea]. Disponible: https://www.advanced-monolithic.com/. 

[12] Espressif Systems, "ESP8266 Deep-sleep and Low Power Solutions," App Notes. [En linea]. Disponible: https://docs.espressif.com/. 

[13] TDK InvenSense, "Low-Power Accelerometer Operating Modes (Application Note)," s. f. [En linea]. Disponible: https://invensense.tdk.com/. 

[14] Proyecto Node-RED, "Node-RED Dashboard (UI) - Documentation." [En linea]. Disponible: https://nodered.org/docs/user-guide/dashboard/. 


## Solicitudes al comite/directores (bloqueadores)
